# Cell-of-origin correlation analysis

For every sample in the pipeline's `results/` folder:

1. loads the per-gene FFT WPS matrix (`results/<sample>/proj/fft_summaries/fft_<sample>_WPS.tsv.gz`, genes x periods 120-280 bp)
2. scores each gene as the mean FFT intensity over 193-199 bp (same as `collapse_fft_wps.py`)
3. correlates the scores with the Tabula Sapiens reference (`resources/avg_expr_by_celltype_tissue.csv`) using several methods
4. plots correlation-vs-period curves ("S curves"): the dip around 193-199 bp for the contributing cell types

Only reads pipeline outputs; everything is written to `OUT_DIR`.

**Sanity check used throughout:** mature erythrocytes have no nucleus and cannot release cfDNA,
so `erythrocyte_*` groups should rank near the bottom. Blood/immune groups are expected at the top.

## Settings: set the paths below

In [ ]:
from pathlib import Path

# the pipeline's workflow/ directory; ".." works when this notebook sits in workflow/result_analysis/
WORKFLOW_DIR = Path("..")

REF_CSV = WORKFLOW_DIR / "resources/avg_expr_by_celltype_tissue.csv"
RESULTS_DIR = WORKFLOW_DIR / "results"

# None = every sample that has an FFT matrix in RESULTS_DIR, or list them, e.g. ["healthy", "cancer"]
SAMPLES = None

# where tables and plots are written
OUT_DIR = Path("output")

BAND = (193, 199)                    # period band (bp) used for the gene score
VAR_GENE_COUNTS = [5000, 2000, 1000, 500]
CURVE_METHOD_N = 5000                # S curves: Pearson on top-N variable genes; None = all genes
TOP_N = 20                           # rows in the top-N tables

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

OUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 45)
pd.set_option("display.width", 250)

def fft_path(sample):
    d = RESULTS_DIR / sample / "proj" / "fft_summaries"
    for name in (f"fft_{sample}_WPS.tsv.gz", f"fft_{sample}_WPS.tsv"):
        if (d / name).exists():
            return d / name
    return None

if SAMPLES is None:
    SAMPLES = sorted(p.name for p in RESULTS_DIR.iterdir() if p.is_dir() and fft_path(p.name))
FFT_FILES = {s: fft_path(s) for s in SAMPLES}
missing = [s for s, p in FFT_FILES.items() if p is None]
assert not missing, f"no FFT matrix found for: {missing}"
assert REF_CSV.exists(), f"reference not found: {REF_CSV}"

print("reference:", REF_CSV)
for s, p in FFT_FILES.items():
    print(f"{s}: {p}")
print("output:", OUT_DIR.resolve())

## Load data

In [ ]:
ref = pd.read_csv(REF_CSV, index_col=0)
gene_names = ref.pop("gene_name")
print(f"reference: {ref.shape[0]} genes x {ref.shape[1]} groups, values {ref.values.min():.2f} .. {ref.values.max():.2f}")

IMMUNE = ["monocyte", "macrophage", "T cell", "B cell", "neutrophil", "dendritic", "NK", "natural killer",
          "lymphocyte", "leukocyte", "myeloid", "plasma cell", "mast cell", "thymocyte", "erythro",
          "hematopoietic", "platelet", "megakaryocyte", "basophil", "eosinophil", "granulocyte", "microglial"]
is_immune = pd.Series([any(k.lower() in g.lower() for k in IMMUNE) for g in ref.columns], index=ref.columns)
ERY = [g for g in ref.columns if g.startswith("erythrocyte_")]
print(f"{is_immune.sum()} groups labelled blood/immune, {len(ERY)} erythrocyte groups")

def load_fft(path):
    fft = pd.read_csv(path, sep="\t", index_col=0)
    fft.columns = fft.columns.astype(float)
    return fft[sorted(fft.columns)]

fft = {s: load_fft(p) for s, p in FFT_FILES.items()}
scores = {}
for s, m in fft.items():
    band_cols = [c for c in m.columns if BAND[0] <= c <= BAND[1]]
    scores[s] = m[band_cols].mean(axis=1)
    shared = ref.index.intersection(m.index)
    print(f"{s}: {m.shape[0]} genes x {m.shape[1]} periods, band columns {band_cols}, "
          f"{len(shared)} genes shared with reference, median score {scores[s].median():.1f}")

In [ ]:
# cross-check the band score against the pipeline's own gene_scores.tsv, where present
for s in SAMPLES:
    gs = RESULTS_DIR / s / "gene_scores.tsv"
    if gs.exists():
        pipe = pd.read_csv(gs, sep="\t", index_col=0).iloc[:, 0]
        common = pipe.index.intersection(scores[s].index)
        print(f"{s}: max abs diff vs gene_scores.tsv = {(pipe[common] - scores[s][common]).abs().max():.2e} "
              f"over {len(common)} genes")
    else:
        print(f"{s}: no gene_scores.tsv to compare against")

## Methods

- **baseline**: Pearson, all shared genes (what the pipeline's `correlate.py` does)
- **pearson_varN**: Pearson on the N genes whose expression varies most across groups; drops genes that are flat
  across all cell types (e.g. housekeeping), which cannot distinguish cell types but add noise
- **spearman_var5k**: Spearman on the top 5000 variable genes. Kept for comparison: rank ties on sparse groups
  (e.g. `erythrocyte_Blood`, ~1000 expressed genes) push them to the top, so it fails the erythrocyte check

In [ ]:
def variable_genes(expr, n):
    return expr.var(axis=1).nlargest(n).index

def pearson(expr, score):
    return expr.corrwith(score)

def spearman(expr, score):
    return expr.rank().corrwith(score.rank())

def label(n):
    return f"pearson_var{n // 1000}k" if n >= 1000 and n % 1000 == 0 else f"pearson_var{n}"

METHODS = {"baseline": lambda e, s: pearson(e, s)}
for n in VAR_GENE_COUNTS:
    METHODS[label(n)] = lambda e, s, n=n: pearson(e.loc[variable_genes(e, n)], s.loc[variable_genes(e, n)])
METHODS["spearman_var5k"] = lambda e, s: spearman(e.loc[variable_genes(e, 5000)], s.loc[variable_genes(e, 5000)])

results = {}  # (method, sample) -> correlation per group, most negative first
for s in SAMPLES:
    shared = ref.index.intersection(scores[s].index)
    expr, score = ref.loc[shared], scores[s].loc[shared]
    for m, f in METHODS.items():
        results[(m, s)] = f(expr, score).sort_values()
print("methods:", list(METHODS))

## Summary per method

- **min_r**: strongest (most negative) correlation; **spread**: max − min over groups
- **immune_topN**: fraction of the top-N groups that are blood/immune
- **ery_blood_rank** / **ery_median_rank**: rank of `erythrocyte_Blood` / median rank of all erythrocyte groups
  (1 = most negative; last = number of groups). Should be near the bottom

In [ ]:
n_groups = ref.shape[1]
rows = []
for m in METHODS:
    for s in SAMPLES:
        r = results[(m, s)]
        rk = r.rank()
        rows.append({"method": m, "sample": s, "min_r": r.min(), "spread": r.max() - r.min(),
                     f"immune_top{TOP_N}": is_immune[r.index[:TOP_N]].mean(), "top1": r.index[0],
                     "ery_blood_rank": int(rk["erythrocyte_Blood"]), "ery_median_rank": rk[ERY].median()})
summary = pd.DataFrame(rows).set_index(["method", "sample"])
if len(SAMPLES) > 1:
    for m in METHODS:
        prof = pd.DataFrame({s: results[(m, s)] for s in SAMPLES})
        summary.loc[m, "mean_rho_between_samples"] = (
            prof.corr(method="spearman").values[np.triu_indices(len(SAMPLES), 1)].mean())
print(f"{n_groups} groups; erythrocyte ranks near {n_groups} are good")
summary.to_csv(OUT_DIR / "method_summary.csv")
summary.round(3)

## Top groups per method (erythrocyte groups marked with `*`)

In [ ]:
for s in SAMPLES:
    top = pd.DataFrame({m: [g + (" *" if g.startswith("erythro") else "") for g in results[(m, s)].index[:TOP_N]]
                        for m in METHODS})
    top.index = range(1, TOP_N + 1)
    top.to_csv(OUT_DIR / f"top{TOP_N}_{s}.csv")
    print(f"=== {s}")
    display(top)

In [ ]:
# all correlations, one column per method x sample
allcorr = pd.DataFrame({f"{m}__{s}": r for (m, s), r in results.items()})
allcorr.index.name = "cell_type"
allcorr.to_csv(OUT_DIR / "correlations_all_methods.csv")
print(f"wrote {allcorr.shape} to {OUT_DIR / 'correlations_all_methods.csv'}")

## Sample vs sample

Each dot is a cell-type group, positioned by its correlation in two samples; dashed line = equal in both.
A tight line means the method is consistent; a whole cloud off the diagonal is a global (likely technical, e.g. depth)
shift; single groups far off the line are candidate sample-specific signal (only meaningful with replicates).

In [ ]:
if len(SAMPLES) >= 2:
    a, b = SAMPLES[0], SAMPLES[1]
    ms = list(METHODS)
    ncol = 3
    fig, axes = plt.subplots(int(np.ceil(len(ms) / ncol)), ncol, figsize=(5 * ncol, 4.5 * np.ceil(len(ms) / ncol)))
    for ax, m in zip(axes.flat, ms):
        x, y = results[(m, a)], results[(m, b)].loc[results[(m, a)].index]
        ax.scatter(x, y, s=8, alpha=0.6, c=np.where(is_immune[x.index], "tab:red", "tab:gray"))
        ery = [g for g in ERY if g in x.index]
        ax.scatter(x[ery], y[ery], s=30, marker="x", c="black", label="erythrocyte")
        lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
        ax.plot([lo, hi], [lo, hi], "k--", lw=0.8)
        ax.set_title(m); ax.set_xlabel(f"{a} r"); ax.set_ylabel(f"{b} r")
    for ax in list(axes.flat)[len(ms):]:
        ax.axis("off")
    axes.flat[0].legend(fontsize=8)
    fig.suptitle(f"per-group correlation, {a} vs {b} (red = blood/immune, x = erythrocyte)")
    plt.tight_layout()
    fig.savefig(OUT_DIR / f"scatter_{a}_vs_{b}.png", dpi=150)
    plt.show()
else:
    print("only one sample, skipping")

## Correlation vs period curves (S curves)

For every period column (120-280 bp), correlate the per-gene FFT intensity at that period with each group's expression.
Contributing cell types should dip around 193-199 bp (shaded); non-contributing ones (erythrocyte, hepatocyte,
keratinocyte) should stay flat around 0. Thick lines are the averages over blood/immune and over all other groups.

In [ ]:
def period_curves(m, n_var=None):
    genes = ref.index.intersection(m.index)
    if n_var:
        genes = variable_genes(ref.loc[genes], n_var)
    expr = ref.loc[genes]
    return pd.DataFrame({p: expr.corrwith(m.loc[genes, p]) for p in m.columns}).T  # periods x groups

curves = {s: period_curves(fft[s], CURVE_METHOD_N) for s in SAMPLES}
for s, c in curves.items():
    c.to_csv(OUT_DIR / f"period_curves_{s}.csv")
print(f"curves: {len(next(iter(curves.values())))} periods x {ref.shape[1]} groups per sample")

In [ ]:
CONTROLS = ["erythrocyte_Blood", "hepatocyte_Liver", "keratinocyte_Tongue"]
N_TOP_CURVES = 4

def plot_curves(curve, title):
    band = curve.loc[(curve.index >= BAND[0]) & (curve.index <= BAND[1])].mean()
    groups = list(band.nsmallest(N_TOP_CURVES).index) + [g for g in CONTROLS if g in curve.columns]
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.axvspan(*BAND, color="gold", alpha=0.25, label=f"{BAND[0]}-{BAND[1]} bp")
    ax.axhline(0, color="k", lw=0.6)
    ax.plot(curve.index, curve.loc[:, is_immune[curve.columns]].mean(axis=1), color="tab:red", lw=3, alpha=0.35,
            label="mean blood/immune groups")
    ax.plot(curve.index, curve.loc[:, ~is_immune[curve.columns]].mean(axis=1), color="tab:gray", lw=3, alpha=0.35,
            label="mean other groups")
    for g in groups:
        style = dict(ls="--", color="black") if g.startswith("erythrocyte") else {}
        ax.plot(curve.index, curve[g], lw=1.4, label=g, **style)
    ax.set_xlabel("period (bp)"); ax.set_ylabel("correlation with expression"); ax.set_title(title)
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    return fig

for s, c in curves.items():
    fig = plot_curves(c, f"{s}: Pearson, top {CURVE_METHOD_N or 'all'} variable genes")
    fig.savefig(OUT_DIR / f"scurve_{s}.png", dpi=150)
    plt.show()

In [ ]:
# all samples on one plot: mean blood/immune curve vs mean other groups, and erythrocyte_Blood
fig, ax = plt.subplots(figsize=(11, 5))
ax.axvspan(*BAND, color="gold", alpha=0.25)
ax.axhline(0, color="k", lw=0.6)
for i, (s, c) in enumerate(curves.items()):
    col = plt.cm.tab10(i)
    ax.plot(c.index, c.loc[:, is_immune[c.columns]].mean(axis=1), color=col, lw=2, label=f"{s}: blood/immune mean")
    ax.plot(c.index, c.loc[:, ~is_immune[c.columns]].mean(axis=1), color=col, lw=1, ls=":", label=f"{s}: other mean")
    ax.plot(c.index, c["erythrocyte_Blood"], color=col, lw=1, ls="--", label=f"{s}: erythrocyte_Blood")
ax.set_xlabel("period (bp)"); ax.set_ylabel("correlation with expression")
ax.set_title(f"S curves across samples (Pearson, top {CURVE_METHOD_N or 'all'} variable genes)")
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
fig.savefig(OUT_DIR / "scurve_all_samples.png", dpi=150)
plt.show()